In [0]:
%pip install -q torch scikit-learn scipy

In [0]:
%run ../utils/utils

## Modelo — Autoencoder para Anomalias de Vendas

 Responsabilidade ÚNICA deste notebook: treinar o modelo e avaliar sua
 qualidade. A ENTRADA é exclusivamente `stg_features_engineered`, gerada
 pelo notebook `feature_engineering.py` — nenhuma coluna nova é criada
 aqui, só tratamento estatístico (split, escala, treino, avaliação).

In [0]:
 
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from scipy import stats
import torch
import torch.nn as nn
 
torch.manual_seed(42)
np.random.seed(42)
 
CAMINHO_MODELO = "IA/encouders/autoencoder_pytorch.pt"
CAMINHO_SCALER = "IA/encouders/scaler.joblib"
CAMINHO_METADADOS = "IA/encouders/metadados.json"

## Ler a entrada oficial (saída do feature_engineering.py)

In [0]:
df_features_final = ler_delta("IA/encouders", "stg_features_engineered", STORAGE_OPTIONS)
print(f"Pedidos recebidos do feature_engineering: {df_features_final.count()}")

colunas_onehot = sorted([c for c in df_features_final.columns if c.startswith("pgto_")])
colunas_features = COLUNAS_NUMERICAS_ANOMALIA + colunas_onehot
print(f"Colunas de feature (na ordem usada pelo modelo): {colunas_features}")


## Split temporal (80/20)
 Por que temporal e não aleatório: as features de histórico do cliente
 (`ticket_medio_historico_cliente`, `dias_desde_ultimo_pedido`) já dependem
 de ordem cronológica — misturar aleatoriamente vazaria informação do
 "futuro" para o treino.

In [0]:

PERCENTUAL_TREINO = 0.80

df_com_indice = df_features_final.withColumn(
    "indice_temporal", F.row_number().over(Window.orderBy("dt_pedido"))
)

qtd_total = df_com_indice.count()
qtd_treino = int(qtd_total * PERCENTUAL_TREINO)

df_treino_spark = df_com_indice.filter(F.col("indice_temporal") <= qtd_treino).drop("indice_temporal")
df_teste_spark = df_com_indice.filter(F.col("indice_temporal") > qtd_treino).drop("indice_temporal")

print(f"Treino: {df_treino_spark.count()} pedidos | Teste: {df_teste_spark.count()} pedidos")

##  Normalização (StandardScaler) — ajustada SÓ no treino

In [0]:
pdf_treino = df_treino_spark.toPandas()
pdf_teste = df_teste_spark.toPandas()

X_treino = pdf_treino.reindex(columns=colunas_features, fill_value=0).fillna(0).values.astype("float32")
X_teste = pdf_teste.reindex(columns=colunas_features, fill_value=0).fillna(0).values.astype("float32")

scaler = StandardScaler()
X_treino_scaled = scaler.fit_transform(X_treino).astype("float32")
X_teste_scaled = scaler.transform(X_teste).astype("float32")  # transform, nunca fit_transform

qtd_features = X_treino_scaled.shape[1]
print(f"Dimensão de entrada: {qtd_features} features")

## Arquitetura do Autoencoder

In [0]:
dim_entrada = qtd_features
dim_oculta1 = max(8, dim_entrada // 2)
dim_gargalo = max(3, dim_entrada // 4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
autoencoder = AutoencoderTorch(dim_entrada, dim_oculta1, dim_gargalo).to(device)
print(autoencoder)

## Treinamento

In [0]:
n_val = int(len(X_treino_scaled) * 0.15)
indices = np.random.permutation(len(X_treino_scaled))
idx_val, idx_treino_real = indices[:n_val], indices[n_val:]

X_treino_real = torch.tensor(X_treino_scaled[idx_treino_real]).to(device)
X_val = torch.tensor(X_treino_scaled[idx_val]).to(device)

otimizador = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)
funcao_perda = nn.MSELoss()

EPOCAS, BATCH_SIZE, PATIENCE = 100, 64, 10
melhor_val_loss = float("inf")
epocas_sem_melhora = 0
melhores_pesos = None
historico_loss_treino, historico_val_loss = [], []

dataset_treino = torch.utils.data.TensorDataset(X_treino_real, X_treino_real)
loader_treino = torch.utils.data.DataLoader(dataset_treino, batch_size=BATCH_SIZE, shuffle=True)

for epoca in range(EPOCAS):
    autoencoder.train()
    perda_acumulada = 0.0
    for batch_x, batch_y in loader_treino:
        otimizador.zero_grad()
        reconstrucao = autoencoder(batch_x)
        perda = funcao_perda(reconstrucao, batch_y)
        perda.backward()
        otimizador.step()
        perda_acumulada += perda.item() * batch_x.size(0)

    loss_treino = perda_acumulada / len(dataset_treino)

    autoencoder.eval()
    with torch.no_grad():
        val_loss = funcao_perda(autoencoder(X_val), X_val).item()

    historico_loss_treino.append(loss_treino)
    historico_val_loss.append(val_loss)
    print(f"Epoch {epoca+1}/{EPOCAS} - loss: {loss_treino:.4f} - val_loss: {val_loss:.4f}")

    if val_loss < melhor_val_loss:
        melhor_val_loss = val_loss
        epocas_sem_melhora = 0
        melhores_pesos = {k: v.clone() for k, v in autoencoder.state_dict().items()}
    else:
        epocas_sem_melhora += 1
        if epocas_sem_melhora >= PATIENCE:
            print(f"EarlyStopping na época {epoca+1}.")
            break

autoencoder.load_state_dict(melhores_pesos)

###  Gráfico 1 — Curva de aprendizado (loss treino x validação)

  Se o modelo está aprendendo de verdade, a loss de treino e validação
  devem cair juntas e estabilizar. Se a validação sobe enquanto o treino
  cai (overfitting), o EarlyStopping deve ter interrompido antes disso —
  a linha vertical marca onde os melhores pesos foram salvos.

In [0]:
epoca_melhor = int(np.argmin(historico_val_loss)) + 1
 
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, len(historico_loss_treino) + 1), historico_loss_treino, label="Loss (treino)", color="#4C72B0")
ax.plot(range(1, len(historico_val_loss) + 1), historico_val_loss, label="Loss (validação)", color="#DD8452")
ax.axvline(epoca_melhor, color="#55A868", linestyle="--", label=f"Melhor época ({epoca_melhor})")
ax.set_xlabel("Época")
ax.set_ylabel("MSE (loss)")
ax.set_title("Curva de Aprendizado do Autoencoder")
ax.legend()
plt.tight_layout()
plt.show()
 
print(f"Loss caiu de {historico_loss_treino[0]:.4f} (época 1) para {historico_loss_treino[-1]:.4f} "
      f"(última época treinada) — treino. Validação: {historico_val_loss[0]:.4f} -> {min(historico_val_loss):.4f}.")

## Limiar de anomalia e score no conjunto de teste

In [0]:
PERCENTIL_LIMIAR = 95

autoencoder.eval()
with torch.no_grad():
    reconstrucao_treino = autoencoder(torch.tensor(X_treino_scaled).to(device)).cpu().numpy()
    reconstrucao_teste = autoencoder(torch.tensor(X_teste_scaled).to(device)).cpu().numpy()

erro_treino = np.mean(np.square(X_treino_scaled - reconstrucao_treino), axis=1)
erro_teste = np.mean(np.square(X_teste_scaled - reconstrucao_teste), axis=1)
limiar_anomalia = float(np.percentile(erro_treino, PERCENTIL_LIMIAR))

pdf_teste["erro_reconstrucao"] = erro_teste
pdf_teste["is_anomaly"] = erro_teste > limiar_anomalia

print(f"Limiar (p{PERCENTIL_LIMIAR}): {limiar_anomalia:.4f}")
print(f"Anomalias no teste: {pdf_teste['is_anomaly'].sum()} de {len(pdf_teste)}")

###  Gráfico 2 — Distribuição do erro de reconstrução e o limiar
 Mostra visualmente por que o limiar foi colocado ali: a maioria dos
 pedidos (normais) tem erro baixo e concentrado; a cauda à direita do
 limiar é o que vira anomalia. Se o histograma não tivesse uma cauda
 clara (fosse uniforme), o corte seria arbitrário — aqui não é.

In [0]:

#  Escala linear esconde tudo: o erro varia de perto de zero até dezenas/
#  centenas em poucos outliers extremos, então num eixo linear tudo fica
#  espremido perto do zero. Escala LOG no eixo X resolve isso mantendo os
#  dados reais (sem cortar nenhum ponto do gráfico).
epsilon = 1e-6
erro_treino_plot = np.clip(erro_treino, epsilon, None)
erro_teste_plot = np.clip(erro_teste, epsilon, None)
limiar_plot = max(limiar_anomalia, epsilon)
 
valor_min = min(erro_treino_plot.min(), erro_teste_plot.min())
valor_max = max(erro_treino_plot.max(), erro_teste_plot.max())
bins_log = np.logspace(np.log10(valor_min), np.log10(valor_max), 50)
 
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(erro_treino_plot, bins=bins_log, alpha=0.6, label="Treino", color="#4C72B0", density=True)
ax.hist(erro_teste_plot, bins=bins_log, alpha=0.6, label="Teste", color="#DD8452", density=True)
ax.axvline(limiar_plot, color="#C44E52", linestyle="--", linewidth=2,
           label=f"Limiar p{PERCENTIL_LIMIAR} ({limiar_anomalia:.3f})")
ax.set_xscale("log")
ax.set_xlabel("Erro de reconstrução (MSE) — escala log")
ax.set_ylabel("Densidade")
ax.set_title("Distribuição do Erro de Reconstrução e Limiar de Anomalia")
ax.legend()
plt.tight_layout()
plt.show()


## 7. Avaliação do modelo (o quanto ele está correto)

### Por que não existe "acurácia" tradicional aqui

 Não temos rótulo de verdade ("essa compra era anômala: sim/não") — é um
 problema não-supervisionado por natureza. Acurácia/precisão/recall
 exigiriam esse rótulo, que não existe. Em vez disso, usamos 3 métodos de
 avaliação apropriados para este tipo de modelo:

 1. **Erro de reconstrução (loss)** — quanto menor, melhor o modelo aprendeu
    o padrão comum dos dados. É o equivalente mais direto de "o modelo
    está ajustado" para um autoencoder.
 2. **Teste de hipótese (separação estatística)** — confirma que os grupos
    "normal" e "anomalia" são estatisticamente diferentes de verdade, não
    uma separação arbitrária/ruído.
 3. **Comparação contra uma baseline simples (Z-score)** — mostra se o
    autoencoder captura mais nuance que um método estatístico básico.


In [0]:
# --- 7.1 Métricas de erro de reconstrução ---
variancia_dados_teste = np.var(X_teste_scaled)
r2_like = 1 - (np.mean(erro_teste) / variancia_dados_teste)  # análogo a R², não é R² formal

print("===== 7.1 MÉTRICAS DE ERRO DE RECONSTRUÇÃO =====")
print(f"MSE médio no treino:  {np.mean(erro_treino):.4f}")
print(f"MSE médio no teste:   {np.mean(erro_teste):.4f}")
print(f"Melhor val_loss (treino): {melhor_val_loss:.4f}")
print(f"'Variância explicada' aproximada (análogo a R²): {r2_like:.2%}")
print("(quanto mais perto de 100%, melhor o modelo reconstrói o padrão normal)")

In [0]:
# --- 7.2 Teste de hipótese: os grupos são estatisticamente diferentes? ---
grupo_normal = pdf_teste[pdf_teste["is_anomaly"] == False]
grupo_anomalia = pdf_teste[pdf_teste["is_anomaly"] == True]

resultados_teste_hipotese = []
for feat in COLUNAS_NUMERICAS_ANOMALIA:
    v_normal = grupo_normal[feat].dropna()
    v_anomalia = grupo_anomalia[feat].dropna()
    if len(v_anomalia) < 2:
        continue
    _, p_ttest = stats.ttest_ind(v_anomalia, v_normal, equal_var=False, nan_policy="omit")
    _, p_mw = stats.mannwhitneyu(v_anomalia, v_normal, alternative="two-sided")
    resultados_teste_hipotese.append({
        "feature": feat,
        "p_valor_ttest": p_ttest,
        "p_valor_mannwhitney": p_mw,
        "significativo_5pct": (p_ttest < 0.05) and (p_mw < 0.05),
    })

df_teste_hipotese = pd.DataFrame(resultados_teste_hipotese).sort_values("p_valor_ttest")
qtd_significativas = df_teste_hipotese["significativo_5pct"].sum()

print("\n===== 7.2 TESTE DE HIPÓTESE: NORMAL vs ANOMALIA =====")
display(df_teste_hipotese)
print(f"{qtd_significativas} de {len(df_teste_hipotese)} features mostram diferença estatisticamente "
      f"significativa (p < 0.05) entre os grupos — evidência de que o modelo capturou um padrão real.")

###  Gráfico 3 — O modelo separa normal de anomalia?
 Boxplot do erro de reconstrução por grupo (deve haver pouca sobreposição
 se o modelo está funcionando) + gráfico de significância (-log10 do
 p-valor) por feature, com a linha de corte em p=0.05.

In [0]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
 
# 3a. Boxplot erro de reconstrução: normal x anomalia
#  Outliers extremos (ex: um pedido com erro ~140) esmagavam a escala linear
#  e escondiam a distribuição inteira perto do zero. Escala LOG no eixo Y
#  mantém os outliers visíveis sem achatar o resto.
epsilon = 1e-6
dados_box = [
    grupo_normal["erro_reconstrucao"].clip(lower=epsilon),
    grupo_anomalia["erro_reconstrucao"].clip(lower=epsilon),
]
axes[0].boxplot(dados_box, tick_labels=["Normal", "Anomalia"])
axes[0].axhline(max(limiar_anomalia, epsilon), color="#C44E52", linestyle="--",
                label=f"Limiar ({limiar_anomalia:.3f})")
axes[0].set_yscale("log")
axes[0].set_ylabel("Erro de reconstrução (MSE) — escala log")
axes[0].set_title("Erro de Reconstrução: Normal x Anomalia")
axes[0].legend()
 
# 3b. Significância estatística por feature
#  Uma feature com p-valor ~10^-250 gera uma barra gigante que esmaga a
#  comparação entre as demais. Aqui limitamos a barra a um teto visual
#  (LIMITE_LOG10_P) e anotamos o p-valor real ao lado — assim dá pra
#  comparar as features "normais" e ainda saber que a primeira é extrema.
LIMITE_LOG10_P = 20
 
df_plot_p = df_teste_hipotese.copy()
df_plot_p["neg_log10_p"] = -np.log10(df_plot_p["p_valor_ttest"].clip(lower=1e-300))
df_plot_p["neg_log10_p_exibicao"] = df_plot_p["neg_log10_p"].clip(upper=LIMITE_LOG10_P)
df_plot_p = df_plot_p.sort_values("neg_log10_p", ascending=True)
cores = ["#55A868" if s else "#8C8C8C" for s in df_plot_p["significativo_5pct"]]
 
barras_p = axes[1].barh(df_plot_p["feature"], df_plot_p["neg_log10_p_exibicao"], color=cores)
axes[1].axvline(-np.log10(0.05), color="#C44E52", linestyle="--", label="p = 0.05")
axes[1].set_xlim(0, LIMITE_LOG10_P * 1.35)
axes[1].set_xlabel(f"-log10(p-valor) — truncado em {LIMITE_LOG10_P}")
axes[1].set_title("Significância por Feature (verde = significativo)")
axes[1].legend(loc="lower right")
 
for barra, p_real, neg_log_real in zip(barras_p, df_plot_p["p_valor_ttest"], df_plot_p["neg_log10_p"]):
    rotulo = f"p<1e-{LIMITE_LOG10_P}" if neg_log_real > LIMITE_LOG10_P else f"p={p_real:.1e}"
    axes[1].text(barra.get_width(), barra.get_y() + barra.get_height() / 2, f" {rotulo}",
                 va="center", fontsize=8)
 
plt.tight_layout()
plt.show()
 

In [0]:
# --- 7.3 Comparação contra baseline simples (Z-score em valor_total) ---
media_valor_treino = pdf_treino["valor_total"].mean()
desvio_valor_treino = pdf_treino["valor_total"].std()
 
z_score_teste = (pdf_teste["valor_total"] - media_valor_treino) / desvio_valor_treino
baseline_anomaly = z_score_teste.abs() > 2  # heurística simples: |z| > 2
 
sobreposicao = (baseline_anomaly & pdf_teste["is_anomaly"]).sum()
so_baseline = (baseline_anomaly & ~pdf_teste["is_anomaly"]).sum()
so_autoencoder = (~baseline_anomaly & pdf_teste["is_anomaly"]).sum()
 
print("\n===== 7.3 COMPARAÇÃO COM BASELINE (Z-score em valor_total) =====")
print(f"Baseline simples (só valor_total) encontrou: {baseline_anomaly.sum()} anomalias")
print(f"Autoencoder (todas as features) encontrou:   {pdf_teste['is_anomaly'].sum()} anomalias")
print(f"Sobreposição (ambos concordam): {sobreposicao}")
print(f"Só o baseline capturou: {so_baseline}")
print(f"Só o autoencoder capturou: {so_autoencoder} "
      f"(evidência de que o modelo enxerga padrões além do valor bruto — combinação de várias features)")

###  Gráfico 4 — Autoencoder x Baseline (Z-score)
 Se o autoencoder só reproduzisse o que um Z-score simples em
 `valor_total` já faz, "Só o autoencoder" seria ~0 — ou seja, o modelo
 não estaria agregando valor. A barra "Só o autoencoder" grande é o que
 justifica usar uma rede neural em vez de uma regra estatística simples.

In [0]:
categorias_comparacao = ["Só baseline", "Sobreposição\n(ambos concordam)", "Só autoencoder"]
valores_comparacao = [so_baseline, sobreposicao, so_autoencoder]
 
fig, ax = plt.subplots(figsize=(8, 5))
barras = ax.bar(categorias_comparacao, valores_comparacao, color=["#DD8452", "#55A868", "#4C72B0"])
ax.set_ylabel("Qtd. de pedidos")
ax.set_title("Anomalias: Autoencoder x Baseline (Z-score)")
for barra, v in zip(barras, valores_comparacao):
    ax.text(barra.get_x() + barra.get_width() / 2, v, str(v), ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Qual feature mais contribuiu para cada anomalia

In [0]:
erro_por_feature_teste = np.square(X_teste_scaled - reconstrucao_teste)
indice_feature_dominante = np.argmax(erro_por_feature_teste, axis=1)
pdf_teste["feature_dominante"] = [colunas_features[i] for i in indice_feature_dominante]
pdf_teste["erro_feature_dominante"] = erro_por_feature_teste[
    np.arange(len(erro_por_feature_teste)), indice_feature_dominante
]
print("===== Amostra: feature que mais contribuiu para o erro, por pedido =====")
display(
    pdf_teste[["id_pedido", "is_anomaly", "feature_dominante", "erro_feature_dominante", "erro_reconstrucao"]]
    .sort_values("erro_reconstrucao", ascending=False)
    .head(10)
)
 
contagem_dominante_anomalias = (
    pdf_teste.loc[pdf_teste["is_anomaly"], "feature_dominante"]
    .value_counts()
    .reset_index()
)
contagem_dominante_anomalias.columns = ["feature_dominante", "qtd_anomalias"]
 
print(f"\n===== Feature dominante entre as {pdf_teste['is_anomaly'].sum()} anomalias do teste =====")
display(contagem_dominante_anomalias)

## 9. Salvar modelo, scaler, metadados (incluindo métricas de avaliação)

In [0]:
import json
 
salvar_modelo_pytorch(autoencoder, CAMINHO_MODELO, container_squad1)
salvar_modelo_ml(scaler, CAMINHO_SCALER, container_squad1)
 
metadados = {
    "colunas_features": colunas_features,
    "limiar_anomalia": limiar_anomalia,
    "percentil_limiar": PERCENTIL_LIMIAR,
    "fontes_dados": ["squad1", "squad3"],
    "dim_entrada": dim_entrada,
    "dim_oculta1": dim_oculta1,
    "dim_gargalo": dim_gargalo,
    "melhor_val_loss": melhor_val_loss,
    "qtd_epocas_treinadas": len(historico_loss_treino),
    "avaliacao_mse_treino": float(np.mean(erro_treino)),
    "avaliacao_mse_teste": float(np.mean(erro_teste)),
    "avaliacao_variancia_explicada_aprox": float(r2_like),
    "avaliacao_qtd_features_significativas": int(qtd_significativas),
    "avaliacao_sobreposicao_baseline": int(sobreposicao),
}
file_client = container_squad1.get_file_client(CAMINHO_METADADOS)
file_client.upload_data(json.dumps(metadados, ensure_ascii=False, indent=2), overwrite=True)
 
df_historico_loss = spark.createDataFrame(
    [(i + 1, historico_loss_treino[i], historico_val_loss[i]) for i in range(len(historico_loss_treino))],
    ["epoca", "loss_treino", "val_loss"]
)
gravar_delta(df=df_historico_loss, camada="IA/encouders", tabela="stg_treino_historico_loss",
             storage_opts=STORAGE_OPTIONS, mode="overwrite", particionar=False)
 
# Previsões do teste — inclui status_pedido (só interpretativo, nunca foi
# feature) para os insights cruzarem anomalia x pedidos cancelados.
colunas_para_salvar = ["id_pedido", "id_cliente", "origem_squad", "status_pedido",
                        "valor_total", "valor_frete", "qtd_skus_distintos", "desconto_total",
                        "desvio_pct_vs_media_cliente", "hora_do_dia", "qtd_itens",
                        "dias_desde_ultimo_pedido", "qtd_pedidos_anteriores_cliente",
                        "ticket_medio_historico_cliente",
                        "feature_dominante", "erro_feature_dominante", "erro_reconstrucao",
                        "is_anomaly"] + COLUNAS_NUMERICAS_ANOMALIA
colunas_para_salvar = list(dict.fromkeys([c for c in colunas_para_salvar if c in pdf_teste.columns]))
 
df_teste_resultado = spark.createDataFrame(pdf_teste[colunas_para_salvar])
gravar_delta(df=df_teste_resultado, camada="IA/encouders", tabela="stg_teste_predicoes",
             storage_opts=STORAGE_OPTIONS, mode="overwrite", particionar=False)
 
print("Modelo, scaler, metadados (com métricas de avaliação), histórico de loss e previsões salvos.")
print("O notebook 'insights_modelo.py' está pronto para ler esses resultados.")